# 00 · Preparar el entorno y los datos

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

Este notebook es el primero que hay que correr. Hace cuatro cosas:

1. Verifica que el entorno tenga lo necesario.
2. Descarga los datos del CMS, si no están ya.
3. Construye la base SQLite ejecutando los scripts de `datos/`.
4. Comprueba que la base responda y recorre las ocho trampas del dato.

No hay nada de LangChain todavía. Esto es la plomería: se corre una vez y no se
vuelve a tocar.

> **Sobre los datos.** Son reclamos médicos **sintéticos** del CMS de Estados
> Unidos. El organismo alteró deliberadamente las relaciones entre variables para
> proteger la privacidad. Sirven para construir y probar herramientas; **no
> admiten inferencia sobre la población real de Medicare.**

## Legibilidad para proyección

Esta celda agranda la tipografía de las salidas. En una sala proyectada, el
tamaño por defecto no se lee desde la cuarta fila.

In [1]:
from IPython.display import HTML, display

display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

Tipografía ajustada para proyección.


## Dónde estamos

Todas las rutas del proyecto son **relativas a la raíz del repositorio**, nunca
absolutas: así el notebook corre igual en otra máquina. Esta celda resuelve la
raíz y la deja en `RAIZ`.

In [2]:
import sys
from pathlib import Path

# La raíz es la carpeta que contiene datos/. Buscamos hacia arriba desde el cwd,
# para que el notebook funcione aunque Jupyter se haya abierto en otro nivel.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
if RAIZ is None:
    raise SystemExit("No encuentro la carpeta datos/. Abre Jupyter en la raíz del repositorio.")

DATOS = RAIZ / "datos"
BASE = DATOS / "reclamos.db"

# datos/ tiene los scripts del pipeline; hay que poder importarlos.
if str(DATOS) not in sys.path:
    sys.path.insert(0, str(DATOS))

print(f"Raíz del proyecto : {RAIZ}")
print(f"Scripts de datos  : {DATOS}")
print(f"Base de datos     : {BASE}  {'(existe)' if BASE.exists() else '(aún no existe)'}")

Raíz del proyecto : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend
Scripts de datos  : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos
Base de datos     : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos\reclamos.db  (aún no existe)


## Verificar el entorno

Antes de construir nada, revisamos qué hay y qué falta. La idea es que si algo
no está, se sepa **ahora** y no en el minuto 26 de la charla.

El pipeline de datos solo necesita `sqlite3` y `csv`, que vienen con Python, más
`requests` para la descarga. Lo demás es para los notebooks que siguen.

In [3]:
import importlib.util
import sqlite3

print(f"Python {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("  ADVERTENCIA: el proyecto se escribió para 3.11+")
print(f"SQLite {sqlite3.sqlite_version}\n")

# (módulo a importar, paquete en pip, para qué sirve, imprescindible ahora)
PAQUETES = [
    ("requests",     "requests",        "descargar los datos del CMS",      True),
    ("yaml",         "pyyaml",          "leer el diccionario de negocio",   True),
    ("pandas",       "pandas",          "mostrar resultados como tablas",   False),
    ("dotenv",       "python-dotenv",   "leer el .env con las claves",      False),
    ("langchain",    "langchain",       "notebook 02",                      False),
    ("langgraph",    "langgraph",       "notebook 03",                      False),
    ("langsmith",    "langsmith",       "notebook 04",                      False),
    ("fastapi",      "fastapi",         "notebook 05",                      False),
    ("ipywidgets",   "ipywidgets",      "la caja de preguntas en vivo",     False),
]

faltan_ahora, faltan_despues = [], []
for modulo, paquete, para_que, imprescindible in PAQUETES:
    hay = importlib.util.find_spec(modulo) is not None
    marca = "  ok  " if hay else ("FALTA " if imprescindible else " --   ")
    print(f"{marca} {paquete:<16} {para_que}")
    if not hay:
        (faltan_ahora if imprescindible else faltan_despues).append(paquete)

print()
if faltan_ahora:
    print("Falta lo imprescindible. Instala y vuelve a correr esta celda:")
    print(f"  pip install {' '.join(faltan_ahora)}")
elif faltan_despues:
    print("Lo imprescindible está. Para los notebooks que siguen te faltará:")
    print(f"  pip install {' '.join(faltan_despues)}")
    print("  (o directamente:  pip install -r requirements.txt)")
else:
    print("Entorno completo.")

Python 3.13.10
SQLite 3.50.4

FALTA  requests         descargar los datos del CMS
FALTA  pyyaml           leer el diccionario de negocio
 --    pandas           mostrar resultados como tablas
 --    python-dotenv    leer el .env con las claves
 --    langchain        notebook 02
 --    langgraph        notebook 03
 --    langsmith        notebook 04
 --    fastapi          notebook 05
 --    ipywidgets       la caja de preguntas en vivo

Falta lo imprescindible. Instala y vuelve a correr esta celda:
  pip install requests pyyaml


## Las credenciales del modelo

El `.env` no hace falta para construir la base, pero sí desde el notebook 01.
Lo revisamos ahora para no descubrir que falta más adelante.

**El `.env` nunca va al repositorio.** Está en el `.gitignore`. Lo que sí va es
`.env.example`, con los nombres de las variables y sin los valores.

In [6]:
env = RAIZ / ".env"
if not env.exists():
    print("No hay .env todavía. No importa para esta celda ni para construir la base.")
    print("Antes del notebook 01: copia .env.example a .env y pon tus claves.")
else:
    # Leemos solo los nombres de las variables. Los valores no se imprimen nunca:
    # esta salida se commitea al repositorio.
    nombres = [l.split("=", 1)[0].strip() for l in env.read_text(encoding="utf-8").splitlines()
               if l.strip() and not l.strip().startswith("#") and "=" in l]
    print(f".env encontrado, con {len(nombres)} variables definidas:")
    for n in nombres:
        print(f"  {n}")
    print("\n(Los valores no se imprimen: la salida de este notebook va a git.)")

.env encontrado, con 0 variables definidas:

(Los valores no se imprimen: la salida de este notebook va a git.)


## Paso 1 · Descargar los datos del CMS

Tres archivos, 40 MB comprimidos y 193 MB descomprimidos:

| Archivo | Se convierte en |
|---|---|
| Beneficiary Summary 2008 | `afiliados` |
| Inpatient Claims 2008-2010 | `hospitalario` |
| Outpatient Claims 2008-2010 | `ambulatorio` |

La celda es **idempotente**: si los CSV ya están, no vuelve a bajar nada.

Si tu red bloquea `cms.gov` —pasa en redes corporativas y entornos con lista
blanca— la celda te lo dirá. En ese caso baja los tres `.zip` a mano desde la
[página de la muestra 1](https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-1)
y descomprímelos en `datos/extraidos/`.

In [7]:
import descargar

ya_estan = [DATOS / "extraidos" / csv for _, csv, _ in descargar.ARCHIVOS.values()]
if all(p.exists() for p in ya_estan):
    print("Los tres CSV ya están descomprimidos. No hay nada que descargar.\n")
    for p in ya_estan:
        print(f"  {p.name:<52} {p.stat().st_size/1_048_576:>7.1f} MB")
else:
    import os
    anterior = Path.cwd()
    os.chdir(DATOS)          # los scripts trabajan con rutas relativas a datos/
    try:
        if descargar.verificar():
            descargar.descargar()
        else:
            print("\nAlgún enlace del CMS no respondió. Mira el mensaje de arriba.")
    finally:
        os.chdir(anterior)

Los tres CSV ya están descomprimidos. No hay nada que descargar.

  DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv        13.9 MB
  DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv        15.9 MB
  DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv      154.3 MB


## Paso 2 · Construir la base

`preparar.py` hace el trabajo real:

- Traduce las columnas crípticas del CMS a nombres de negocio en español.
- Traduce también **los valores**: sexo, raza, estado de residencia.
- Convierte las fechas y las parte en `anio`, `mes` y `dia`.
- Normaliza los indicadores de condición crónica, que en el origen vienen
  invertidos (1 = sí, 2 = no).
- Clasifica los 11 897 códigos de diagnóstico en los 19 capítulos de la CIE-9.
- Crea los índices y **verifica las ocho trampas** con cifras reales.

Tarda unos 45 segundos y deja `datos/reclamos.db`. La salida de abajo es la
verificación: si algún número no coincide con el README, algo se rompió.

In [8]:
import os
import preparar

anterior = Path.cwd()
os.chdir(DATOS)
try:
    preparar.main(entrada="extraidos", salida="reclamos.db")
finally:
    os.chdir(anterior)

Construyendo reclamos.db desde extraidos/ ...
  cargando afiliados      ... 116,352 filas
  cargando hospitalario   ... 66,773 filas
  cargando ambulatorio    ... 790,790 filas
  cargando capitulos_cie9 ... 19 filas
  derivando diagnosticos ... 2,611,067 filas
  creando índices ...

--- Volúmenes ---
  afiliados         116,352 filas
  hospitalario       66,773 filas
  ambulatorio       790,790 filas
  diagnosticos    2,611,067 filas

--- Trampa 1: crónicos (1=sí, 2=no en el origen) ---
  afiliados con diabetes: 44,060 de 116,352 (37.9%)   [el codebook del CMS dice 37.96%]

--- Trampa 2: no existe 'costo total' ---
  pagado por el asegurador :   639,260,180.00
  deducible del afiliado   :    68,280,716.00
  coaseguro del afiliado   :     6,011,500.00
  pagado por un tercero    :    26,635,700.00
  costo del episodio       :   740,188,096.00  (+15.8%)

--- Trampa 3: montos negativos y en cero ---
  hospitalario       55 negativos (mín -8,000.00),   2,160 en cero
  ambulatorio     2,566 

## Paso 3 · Comprobar que la base responde

Hasta aquí confiamos en lo que imprimió el script. Ahora consultamos la base
nosotros, como lo hará el agente.

In [9]:
import sqlite3

con = sqlite3.connect(BASE)

print("Tablas:\n")
for (nombre,) in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    n = con.execute(f"SELECT count(*) FROM {nombre}").fetchone()[0]
    print(f"  {nombre:<18} {n:>12,} filas")

print(f"\nTamaño del archivo: {BASE.stat().st_size/1_048_576:.1f} MB")

Tablas:

  afiliados               116,352 filas
  ambulatorio             790,790 filas
  capitulos_cie9               20 filas
  diagnosticos          2,611,067 filas
  hospitalario             66,773 filas

Tamaño del archivo: 556.3 MB


### La pregunta fácil

Es la primera que va a responder el agente en el notebook 02. Aquí la hacemos a
mano, para tener el número correcto con el que comparar después.

In [10]:
sql = """
SELECT sum(monto_pagado) AS pagado,
       count(DISTINCT id_reclamo) AS reclamos
FROM hospitalario
WHERE anio = 2009
"""
pagado, reclamos = con.execute(sql).fetchone()
print(f"¿Cuánto se pagó en hospitalización en 2009?\n")
print(f"  Pagado por el asegurador : {pagado:>16,.2f}")
print(f"  Reclamos                 : {reclamos:>16,}")
print(f"  Costo promedio           : {pagado/reclamos:>16,.2f}")

¿Cuánto se pagó en hospitalización en 2009?

  Pagado por el asegurador :   244,810,270.00
  Reclamos                 :           25,231
  Costo promedio           :         9,702.76


### Lo que el agente va a tener que entender

Las columnas están en español y los valores también. Esto es lo que vuelve
viable que un modelo escriba SQL correcto sin adivinar.

In [11]:
for consulta, titulo in [
    ("SELECT sexo, count(*) FROM afiliados GROUP BY sexo ORDER BY 2 DESC",
     "sexo"),
    ("SELECT raza, count(*) FROM afiliados GROUP BY raza ORDER BY 2 DESC",
     "raza"),
    ("SELECT estado, count(*) c FROM afiliados GROUP BY estado ORDER BY c DESC LIMIT 5",
     "estado (los 5 primeros de 52)"),
]:
    print(f"{titulo}:")
    for valor, n in con.execute(consulta):
        print(f"   {valor:<24} {n:>8,}")
    print()

sexo:
   Femenino                   64,347
   Masculino                  52,005

raza:
   Blanca                     96,349
   Negra                      12,343
   Otras                       4,931
   Hispana                     2,729

estado (los 5 primeros de 52):
   California                 10,224
   Florida                     7,745
   Texas                       6,703
   Nueva York                  6,510
   Pensilvania                 5,199



### Los diagnósticos, en lenguaje de negocio

Los 11 897 códigos CIE-9 de la base son ilegibles para una persona. La columna
`grupo_enfermedad` los agrupa en los 19 capítulos de la clasificación, en
español.

**Esta es la diferencia entre un agente que acierta y uno que comunica.**
"El alza está en 41401" no le dice nada a nadie; "el alza está en enfermedades
del aparato circulatorio" sí.

Ojo con el `WHERE orden = 1`: sin ese filtro cada reclamo se cuenta hasta diez
veces, una por cada diagnóstico, y el monto se multiplica.

In [12]:
sql = """
SELECT d.grupo_enfermedad,
       sum(h.monto_pagado) AS pagado,
       count(*)            AS reclamos
FROM diagnosticos d
JOIN hospitalario h ON h.id_reclamo = d.id_reclamo
WHERE d.ambito = 'hospitalario' AND d.orden = 1
GROUP BY d.grupo_enfermedad
ORDER BY pagado DESC
LIMIT 8
"""
print(f"{'Grupo de enfermedad':<58} {'Pagado':>16} {'Reclamos':>10}")
print("-" * 86)
for grupo, pagado, reclamos in con.execute(sql):
    print(f"{grupo:<58} {pagado:>16,.0f} {reclamos:>10,}")

Grupo de enfermedad                                                  Pagado   Reclamos
--------------------------------------------------------------------------------------
Enfermedades del aparato circulatorio                           165,127,000     15,901
Enfermedades del aparato respiratorio                            75,982,000      8,453
Lesiones y envenenamientos                                       67,375,420      6,050
Enfermedades del aparato digestivo                               55,095,130      6,475
Enfermedades del sistema osteomuscular y tejido conjuntivo       53,467,840      4,730
Enfermedades infecciosas y parasitarias                          44,855,820      3,375
Neoplasias                                                       36,521,010      2,920
Factores que influyen en el estado de salud                      32,754,350      2,157


## Paso 4 · Las ocho trampas, una por una

Este es el corazón pedagógico de la charla. **El agente no falla por el modelo:
falla por no saber qué significa cada columna.**

Cada trampa de abajo es una forma en que una consulta correcta en SQL da una
respuesta equivocada en negocio. Todas están documentadas con su verificación en
`datos/diccionario.yaml`, que es lo que el agente va a leer antes de escribir
SQL.

In [13]:
q = lambda s: con.execute(s).fetchone()

print("1 · LOS INDICADORES CRÓNICOS VENÍAN INVERTIDOS")
n, d = q("SELECT count(*), sum(cronico_diabetes) FROM afiliados")
print(f"    Afiliados con diabetes: {d:,} de {n:,} = {d/n:.1%}")
print(f"    El codebook del CMS dice 37.96%. Si saliera 62%, la conversión")
print(f"    está mal: en el origen 1 = sí y 2 = no, no 1/0.\n")

print("2 · NO EXISTE UNA COLUMNA DE COSTO TOTAL")
pag, ded, coa, ter = q("""SELECT sum(monto_pagado), sum(deducible),
                          sum(coaseguro), sum(monto_pagado_tercero)
                          FROM hospitalario""")
tot = pag + ded + coa + ter
print(f"    Pagó el asegurador  : {pag:>16,.2f}   <- 'costo' por defecto")
print(f"    Deducible           : {ded:>16,.2f}")
print(f"    Coaseguro           : {coa:>16,.2f}")
print(f"    Pagó un tercero     : {ter:>16,.2f}")
print(f"    Costo del episodio  : {tot:>16,.2f}   <- {tot/pag-1:+.1%}")
print(f"    Son dos preguntas distintas. El agente debe desambiguar.\n")

print("3 · HAY MONTOS NEGATIVOS Y EN CERO, Y SON VÁLIDOS")
for t in ("hospitalario", "ambulatorio"):
    neg, mn = q(f"SELECT count(*), min(monto_pagado) FROM {t} WHERE monto_pagado < 0")
    cero = q(f"SELECT count(*) FROM {t} WHERE monto_pagado = 0")[0]
    print(f"    {t:<14} {neg:>6,} negativos (mín {mn:,.0f}), {cero:>7,} en cero")
print(f"    Un WHERE monto > 0 puesto por reflejo los borra.\n")

print("4 · EL PERIODO REAL NO ES EL DEL NOMBRE DEL ARCHIVO")
for t in ("hospitalario", "ambulatorio"):
    desde, hasta = q(f"SELECT min(fecha_inicio), max(fecha_inicio) FROM {t}")
    fuera = q(f"SELECT count(*) FROM {t} WHERE anio NOT BETWEEN 2008 AND 2010")[0]
    print(f"    {t:<14} {desde} a {hasta} — {fuera:,} reclamos fuera de 2008-2010")
print(f"    El archivo se llama '2008_to_2010' y trae 2007.\n")

print("5 · CONTAR FILAS NO ES CONTAR RECLAMOS")
for t in ("hospitalario", "ambulatorio"):
    filas, recl = q(f"SELECT count(*), count(DISTINCT id_reclamo) FROM {t}")
    print(f"    {t:<14} {filas:>9,} filas -> {recl:>9,} reclamos "
          f"(sobran {filas-recl:,})")
print(f"    Un reclamo largo viene partido en hasta dos segmentos.\n")

print("6 · UN CUARTO DE LOS AFILIADOS NO TIENE NINGÚN RECLAMO")
sin = q("""SELECT count(*) FROM afiliados a WHERE NOT EXISTS
           (SELECT 1 FROM hospitalario h WHERE h.id_afiliado = a.id_afiliado)
           AND NOT EXISTS
           (SELECT 1 FROM ambulatorio o WHERE o.id_afiliado = a.id_afiliado)""")[0]
tot_afi = q("SELECT count(*) FROM afiliados")[0]
print(f"    {sin:,} de {tot_afi:,} = {sin/tot_afi:.1%}")
print(f"    Un INNER JOIN los desaparece e infla el gasto per cápita.\n")

print("7 · LOS MONTOS ANUALES DE afiliados SON SOLO DE 2008")
anual = q("SELECT sum(pago_anual_hospitalario) FROM afiliados")[0]
r2008 = q("SELECT sum(monto_pagado) FROM hospitalario WHERE anio = 2008")[0]
print(f"    Suma anual en afiliados : {anual:>16,.2f}  (solo 2008)")
print(f"    Reclamos de 2008        : {r2008:>16,.2f}")
print(f"    Las tablas de reclamos cubren tres años. No son comparables.\n")

print("8 · 'OTHER' NO ES UN CÓDIGO DE DIAGNÓSTICO")
n_other = q("SELECT count(*) FROM diagnosticos WHERE codigo = 'OTHER'")[0]
sin_cap = q("SELECT count(*) FROM diagnosticos WHERE capitulo IS NULL")[0]
print(f"    Aparece {n_other:,} veces. Es un marcador que dejó el CMS.")
print(f"    Tiene capítulo 99, 'Sin clasificar', para que no desaparezca")
print(f"    en un GROUP BY. Diagnósticos sin capítulo: {sin_cap}.")

1 · LOS INDICADORES CRÓNICOS VENÍAN INVERTIDOS
    Afiliados con diabetes: 44,060 de 116,352 = 37.9%
    El codebook del CMS dice 37.96%. Si saliera 62%, la conversión
    está mal: en el origen 1 = sí y 2 = no, no 1/0.

2 · NO EXISTE UNA COLUMNA DE COSTO TOTAL
    Pagó el asegurador  :   639,260,180.00   <- 'costo' por defecto
    Deducible           :    68,280,716.00
    Coaseguro           :     6,011,500.00
    Pagó un tercero     :    26,635,700.00
    Costo del episodio  :   740,188,096.00   <- +15.8%
    Son dos preguntas distintas. El agente debe desambiguar.

3 · HAY MONTOS NEGATIVOS Y EN CERO, Y SON VÁLIDOS
    hospitalario       55 negativos (mín -8,000),   2,160 en cero
    ambulatorio     2,566 negativos (mín -100),  30,105 en cero
    Un WHERE monto > 0 puesto por reflejo los borra.

4 · EL PERIODO REAL NO ES EL DEL NOMBRE DEL ARCHIVO
    hospitalario   2007-11-27 a 2010-12-30 — 224 reclamos fuera de 2008-2010
    ambulatorio    2007-12-12 a 2010-12-31 — 312 reclamos fue

## Listo

La base está construida y verificada. De aquí en adelante ya no hay que volver a
`datos/`.

```python
import sqlite3
con = sqlite3.connect("datos/reclamos.db")
```

**Lo que sigue:** `02_langchain/` — la primera herramienta, para que un modelo
pueda consultar esta base en lenguaje natural.

---

### Opcional: las descripciones de los códigos CIE-9

Los 11 897 códigos de diagnóstico quedaron sin descripción a propósito: no hay
catálogo oficial en español, y traducir ese volumen de terminología clínica sin
fuente sería inventar significados médicos.

Para hablar de diagnósticos ya tenemos `grupo_enfermedad` (19 capítulos en
español) y `categoria` (los tres primeros dígitos). Con eso alcanza para la
charla.

Si además quieres el texto oficial código por código, `datos/enriquecer.py` lo
baja del catálogo de la Biblioteca Nacional de Medicina de Estados Unidos. Está
**en inglés**, y requiere red hacia `clinicaltables.nlm.nih.gov`:

```python
import enriquecer
enriquecer.main(BASE)
```

Es opcional. El proyecto funciona sin eso.